# Pairwise alignments

---
## Learning Objectives

1. Conceptually undersand dynamic programming and sequence alignment
1. Implement Smith-Waterman algorithm for local alignment


---
## Background

This week we will be implementing Smith-Waterman. This is a dynamic programming algorithm used for local sequence alignment. 

As a reminder, the scoring for Smith-Waterman only uses the scores from the positions above, left, and above-left of the current position in the matrix as below:

<center><img src="figures/Smith-Waterman_scoring.png"></center>

For traceback, you will need to keep track of the direction of the arrows in a matrix and then begin traceback from the maximum value.

# Smith-Waterman Algorithm Implementation Activity
## Introduction

Pairwise sequence alignment is a fundamental technique in bioinformatics used to compare two biological sequences, such as DNA, RNA, or proteins. It helps identify regions of similarity that may indicate functional, structural, or evolutionary relationships between the sequences.

The brute force method for sequence alignment, which involves comparing every possible alignment, is highly inefficient. For two sequences of length m and n, the time complexity would be \(T(m,n) = O(mn \cdot 2^{m+n})\), making it impractical for longer sequences.

Dynamic programming offers a more efficient solution. It breaks down the problem into smaller subproblems and stores their solutions to avoid redundant computations. This approach reduces the time complexity to \(O(mn)\).

The Smith-Waterman algorithm is a dynamic programming approach for local sequence alignment. It identifies the optimal local alignment between two sequences by comparing all possible pairs of segments from the sequences and finding the best-scoring alignment.

## Activity Overview

In this activity, you will implement the Smith-Waterman algorithm for local sequence alignment. The algorithm consists of two main parts:

Scoring matrix calculation
Traceback for optimal alignment reconstruction

The pseudocode for the Smith-Waterman algorithm is as follows:

$$
\begin{aligned}
& \text{Initialize scoring matrix } H \text{ with zeros} \\
& \text{For } i \text{ from } 1 \text{ to } m: \\
&     \text{For } j \text{ from } 1 \text{ to } n: \\
&         H[i][j] = \max \begin{cases}
&             0 \\
&             H[i-1][j-1] + \text{match/mismatch score} \\
&             H[i-1][j] + \text{gap penalty} \\
&             H[i][j-1] + \text{gap penalty}
&         \end{cases} \\
& \text{Find the highest score in } H \text{ and its position} \\
& \text{Perform traceback from the highest score position} \\
& \text{Return the optimal local alignment}
\end{aligned}
$$

## Instructions

Working in pairs, implement the Smith-Waterman algorithm using Python. Your implementation should include:

1. A function to calculate the scoring matrix (`cal_score`)
2. A function to perform the traceback and reconstruct the optimal alignment (`traceback`)
3. A dynamic scoring strategy that allows for customizable gap penalties and match/mismatch scores (`smith_waterman`)

## Follow these steps:

### Implement the scoring matrix calculation function:

1. Initialize the matrix with zeros
2. Fill the matrix using the Smith-Waterman algorithm
3. Return the completed matrix and the position of the highest score

### Implement the traceback function

1. Start from the highest score position
2. Trace back through the matrix to reconstruct the optimal alignment
3. Return the aligned sequences and the alignment score

### Implement a main function that:

1. Takes two sequences as input
2. Accepts parameters for match score, mismatch penalty, and gap penalty
3. Calls the scoring and traceback functions
4. Prints the aligned sequences and the alignment score

Test your implementation with various input sequences and scoring parameters

---
## Imports

In [1]:
import numpy as np
# Move codes saved in the traceback matrix (same codes as the cal_score docstring)
# 0 = stop, 1 = diagonal, 2 = up, 3 = left
END, DIAG, UP, LEFT = 0, 1, 2, 3

---
## Implement Smith-Waterman algorithm


```
SmithWaterman(seq1, seq2, match, mismatch, gap)
    Initialize [len(seq1)+1] x [len(seq2)+1] numpy array as scoring matrix with first column and row equal to 0
    
    Fill scoring matrix (score_matrix) and Traceback matrix, record the position with max score (max_pos):
    for i in each row number:
        for j in each column number:
            S[i][j] = max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, H[i][j-1] + gap_score, 0 )
            T[i][j] = direction of max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, S[i][j-1] + gap_score, 0 )
    
    Traceback. Find the optimal path through scoring matrix starting at max_pos
    
```

In [2]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back

    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number
        match (int): score added on the diagonal when the two letters are the same
        mismatch (int): score added on the diagonal when the two letters are different
        gap (int): score added for an up or left move (a gap)

    Returns:
        score in position (i,j)
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT
    '''
    #1. Look up a cell's two letters (i-1, j-1 because of the zero border)
    if seq1[i - 1] == seq2[j - 1]:
        letter_score = match
    else:
        letter_score = mismatch

    #2. Calculate score options (diagonal, down, sideway, 0), choose the biggest
    # option 1: up-left box + match/mismatch score
    diag_score = matrix[i - 1][j - 1] + letter_score
    # option 2: box above + gap penalty
    up_score = matrix[i - 1][j] + gap
    # option 3: box to the left + gap penalty
    left_score = matrix[i][j - 1] + gap
    # the box's number is the biggest option; 0 is option 4 (start over)
    score = max(0, diag_score, up_score,
                left_score)

    #3. For each chosen option, note the direction it was calculated from
    # (if tie, then diagonal first, then down, then sideways)
    # if the number is 0 => stop spot (no arrow)
    if score == 0:
        move = END
    # otherwise, check what direction give the winning number.
    elif score == diag_score:
        move = DIAG
    elif score == up_score:
        move = UP
    else:
        move = LEFT

    return score, move
    

In [3]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''Find the optimal path through scoring marix

        diagonal: match/mismatch
        up: letter from seq1, gap in seq2
        left: gap in seq1, letter from seq2

    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from

    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
    '''
    #1. Set up empty string and starting point (cell with the biggest score)
    aligned_seq1 = ""
    aligned_seq2 = ""
    i, j = maximum_position

    #2. Walking until land on a stop spot and read the direction of that cell:
    while traceback_matrix[i][j] != END:
        move = traceback_matrix[i][j]
        # diagonal, aka up-left: pair one letter from each sequence
        if move == DIAG:
            # put this row's letter at the FRONT of aligned_seq1
            aligned_seq1 = seq1[i - 1] + aligned_seq1
            # put this column's letter at the FRONT of aligned_seq2
            aligned_seq2 = seq2[j - 1] + aligned_seq2
            # move one box up and one box left
            i, j = i - 1, j - 1
        # up: use seq1's letter, gap in seq2
        elif move == UP:
            # put this row's letter at the front of aligned_seq1
            aligned_seq1 = seq1[i - 1] + aligned_seq1
            # put a dash at the front of aligned_seq2
            aligned_seq2 = "-" + aligned_seq2
            # move one box up
            i = i - 1
        # left: gap in seq1, use seq2's letter
        elif move == LEFT:
            # put a dash at the front of aligned_seq1
            aligned_seq1 = "-" + aligned_seq1
            # put this column's letter at the front of aligned_seq2
            aligned_seq2 = seq2[j - 1] + aligned_seq2
            j = j - 1

    return aligned_seq1, aligned_seq2
            

In [4]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1, seed=None):
    '''Smith-Waterman algorithm for local alignment

    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +1
        mismatch: default = -1
        gap: default = -1
        seed: default = None (random tie-break every run); give a number to repeat the same pick

    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix
    '''
    # Random pick if several cells tie for the top score
    rng = np.random.default_rng(seed)

    #1. Set up score grid full of 0s; +1 adds the zero border row and column
    score_matrix = np.zeros((len(seq1) + 1, len(seq2) + 1),dtype=int)
    #2. Set up direction grid with same size and full of 0s (0=END)
    traceback_matrix = np.zeros_like(score_matrix)

    #3. Walk through each cell, row by row, top to bottom, left to right
    for i in range(1, len(seq1) + 1):
        for j in range(1, len(seq2) + 1):
            # calculate this cell's score and direction
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch,gap)
            # record calculated score and direction to respective grid
            score_matrix[i][j] = score
            traceback_matrix[i][j] = move

    # 4. Find starting point for traceback
    # In the score grid, find the biggest score
    best_score = score_matrix.max()
    # List every cell (row,column) that has the biggest score
    tied_positions = np.argwhere(score_matrix == best_score)
    # Choose a random index in that list, the associated cell would be the start of the traceback
    pick = rng.integers(len(tied_positions))
    best_position = tuple(int(x) for x in tied_positions[pick])
    # Call traceback function, follow the direction started at best_position
    aligned_seq1, aligned_seq2 = traceback(seq1, seq2, traceback_matrix,
                                           best_position)

    print("Alignment score:", best_score)
    if len(tied_positions) > 1:
        print("Tied top boxes:", len(tied_positions), "- randomly picked",
              best_position)

    return aligned_seq1, aligned_seq2, score_matrix


In [5]:
# Test 1: DNA, provided example from slides
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1,seq2)
aligned_seq1b, aligned_seq2b, score_matrixb = smith_waterman(seq1, seq2, match=2, gap=-2)

print (aligned_seq1)
print (aligned_seq2)
print (aligned_seq1b)
print (aligned_seq2b)
print (score_matrix)


Alignment score: 4
Alignment score: 8
AC-TTA
ACATTA
AC-TTA
ACATTA
[[0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 1 0 1 0 0 2 1]
 [0 1 0 2 1 0 0 1 1]
 [0 0 0 1 1 2 1 0 0]
 [0 0 0 0 0 2 3 2 1]
 [0 0 1 0 1 1 2 4 3]
 [0 0 0 0 0 0 1 3 3]]


In [6]:
# Test 2: DNA, provided example
seq3 = "GGGGAAAAAAAAAAAACCCC"
seq4 = "GGGGAAAAAACCCC"

aligned_seq3, aligned_seq4, score_matrix = smith_waterman(seq3,seq4)


print (aligned_seq3)
print (aligned_seq4)
print (score_matrix)

Alignment score: 10
Tied top boxes: 2 - randomly picked (20, 14)
AAAAAACCCC
AAAAAACCCC
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  1  1  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  2  2  2  1  0  0  0  0  0  0  0  0  0]
 [ 0  1  2  3  3  2  1  0  0  0  0  0  0  0  0]
 [ 0  1  2  3  4  3  2  1  0  0  0  0  0  0  0]
 [ 0  0  1  2  3  5  4  3  2  1  1  0  0  0  0]
 [ 0  0  0  1  2  4  6  5  4  3  2  1  0  0  0]
 [ 0  0  0  0  1  3  5  7  6  5  4  3  2  1  0]
 [ 0  0  0  0  0  2  4  6  8  7  6  5  4  3  2]
 [ 0  0  0  0  0  1  3  5  7  9  8  7  6  5  4]
 [ 0  0  0  0  0  1  2  4  6  8 10  9  8  7  6]
 [ 0  0  0  0  0  1  2  3  5  7  9  9  8  7  6]
 [ 0  0  0  0  0  1  2  3  4  6  8  8  8  7  6]
 [ 0  0  0  0  0  1  2  3  4  5  7  7  7  7  6]
 [ 0  0  0  0  0  1  2  3  4  5  6  6  6  6  6]
 [ 0  0  0  0  0  1  2  3  4  5  6  5  5  5  5]
 [ 0  0  0  0  0  1  2  3  4  5  6  5  4  4  4]
 [ 0  0  0  0  0  0  1  2  3  4  5  7  6  5  5]
 [ 0  0  0  0  0  0  0  1  2  3  4  6  8  7  6]
 

Extra tests: RNA and protein

RNA example: short made-up test sequences (U instead of T).

Protein example: `HEAGAWGHEE` / `PAWHEAE`, a textbook example from Durbin et al., *Biological Sequence Analysis*. Note: our scoring treats every amino-acid mismatch the same (-1); Protein sequence alignment commonly relies on substitution matrices like BLOSUM (Blocks Substitution Matrix) or PAM (Point Accepted Mutation) to score how likely one amino acid is to be replaced by another.

In [7]:
# Test 3: RNA (suggested by AI)
seq5 = "AUGGCUAGCA"
seq6 = "GCUAGCAUGG"

aligned_seq5, aligned_seq6, score_matrix = smith_waterman(seq5,seq6)

print (aligned_seq5)
print (aligned_seq6)
print (score_matrix)


Alignment score: 7
GCUAGCA
GCUAGCA
[[0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 0 0]
 [0 0 0 1 0 0 0 0 2 1 0]
 [0 1 0 0 0 1 0 0 1 3 2]
 [0 1 0 0 0 1 0 0 0 2 4]
 [0 0 2 1 0 0 2 1 0 1 3]
 [0 0 1 3 2 1 1 1 2 1 2]
 [0 0 0 2 4 3 2 2 1 1 1]
 [0 1 0 1 3 5 4 3 2 2 2]
 [0 0 2 1 2 4 6 5 4 3 2]
 [0 0 1 1 2 3 5 7 6 5 4]]


In [8]:
# Test 4: Protein (suggested by AI)
seq7 = "HEAGAWGHEE"
seq8 = "PAWHEAE"

aligned_seq7, aligned_seq8, score_matrix = smith_waterman(seq7,seq8)

print (aligned_seq7)
print (aligned_seq8)
print (score_matrix)

Alignment score: 3
Tied top boxes: 3 - randomly picked (3, 6)
HEA
HEA
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 2 1 1]
 [0 0 1 0 0 1 3 2]
 [0 0 0 0 0 0 2 2]
 [0 0 1 0 0 0 1 1]
 [0 0 0 2 1 0 0 0]
 [0 0 0 1 1 0 0 0]
 [0 0 0 0 2 1 0 0]
 [0 0 0 0 1 3 2 1]
 [0 0 0 0 0 2 2 3]]
